# Tái lập code GCN tác giả: HC3 → GPT-3.5 Mixed, nhiễu 0%

Chạy tuần tự từ trên xuống. Bật **Internet + GPU T4/P100** trên Kaggle.

- Dùng commit tác giả `83ebc6e2cd543477f83e405d702ac837e664fc9e`, không dùng code GCN nhóm.
- Chạy chính `gcn.py`: **seed 2024, batch 1, không shuffle, Adam 1e-4, weight decay 0, dropout 0.5, đủ 40 epoch**. Không sửa lỗi early stopping của tác giả.
- Seed vẫn nằm đúng vị trí trong script gốc: sau khi nạp cache, ngay trước khởi tạo GCN.
- Python 3.8 riêng, Torch 1.11.0+cu113, PyG 2.5.2, Transformers 4.40.0, spaCy 3.7.4; không thay thư viện kernel Kaggle.
- Chỉ sửa đường dẫn, khai báo biến `device` còn thiếu của builder, chọn GPT clean làm test, và xuất report sau tính toán gốc.
- RoBERTa build embedding bằng **GPU CUDA, float32, eval/no_grad**; spaCy vẫn chạy CPU. Mỗi chunk embedding được chuyển về CPU ngay, cache không chứa tensor CUDA. Tối đa **3 cache graph**. Chuyển CPU sang GPU có thể tạo sai khác số học nhỏ.
- Đây là tái lập **code và cấu hình công bố**, không cam kết trùng bit/điểm paper: thiếu cache, hash encoder và phiên bản model spaCy gốc. `en_core_web_sm==3.7.1` là bản tương thích được khóa, không khẳng định tác giả đã dùng bản này.
- Chỉ test **một điều kiện** Table 3 (0%). Có thể đổi SEED, nhưng seed khác 2024 không phải run mặc định của tác giả.
- Không tự xóa run cũ. Nếu chạy lại cùng seed, đổi tên thư mục RUN ở cell đầu.


In [ ]:
from pathlib import Path
import json
import os
import shutil

WORK = Path("/kaggle/working/author_prdetect_gcn_exact")
REPO = WORK / "PRDetect"
ENV = WORK / "python38"
PYTHON = ENV / "bin/python"
GRAPH = WORK / "graph_data"
SEED = 2024
AUTHOR_COMMIT = "83ebc6e2cd543477f83e405d702ac837e664fc9e"
RUN = WORK / f"seed{SEED}"
PREPARE = WORK / "prepare_author_run.py"
BUILD_SCRIPT = RUN / "author_building_graph_portable.py"
TRAIN_SCRIPT = RUN / "author_gcn_portable.py"
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
GRAPH.mkdir(exist_ok=True)
assert not (RUN / "model" / f"hc3_gcn_model_{SEED}.pth").exists(), (
    "Đã có checkpoint: đổi tên RUN để chạy mới; không ghi đè run cũ."
)
os.environ["UV_PYTHON_INSTALL_DIR"] = str(WORK / "managed_python")
print("Seed:", SEED, "| Cache graphs: 3 | Free disk GiB:",
      round(shutil.disk_usage(WORK).free / 2**30, 2))


In [ ]:
# uv chỉ tạo môi trường riêng; kernel hiện tại không cần restart.
%pip install -q uv
assert get_ipython().user_ns.get("_exit_code", 1) == 0
!uv python install 3.8.20
assert get_ipython().user_ns.get("_exit_code", 1) == 0
if not PYTHON.is_file():
    !uv venv --python 3.8.20 --seed "{ENV}"
    assert get_ipython().user_ns.get("_exit_code", 1) == 0

!uv pip install --python "{PYTHON}" --no-cache "https://download.pytorch.org/whl/cu113/torch-1.11.0%2Bcu113-cp38-cp38-linux_x86_64.whl"
assert get_ipython().user_ns.get("_exit_code", 1) == 0
!uv pip install --python "{PYTHON}" --no-cache "numpy==1.22.4" "scipy==1.10.1" "pandas==2.0.3" "scikit-learn==1.3.1" "spacy==3.7.4" "transformers==4.40.0" "huggingface-hub==0.20.1" "tqdm==4.66.1" "tensorboard==2.9.1" "protobuf==3.19.6" "regex==2023.8.8" "requests==2.31.0"
assert get_ipython().user_ns.get("_exit_code", 1) == 0
!uv pip install --python "{PYTHON}" --no-cache --no-index --only-binary=:all: -f "https://data.pyg.org/whl/torch-1.11.0+cu113.html" "torch-scatter==2.0.9" "torch-sparse==0.6.13"
assert get_ipython().user_ns.get("_exit_code", 1) == 0
!uv pip install --python "{PYTHON}" --no-cache "torch-geometric==2.5.2" "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
assert get_ipython().user_ns.get("_exit_code", 1) == 0
!{PYTHON} -m pip check
assert get_ipython().user_ns.get("_exit_code", 1) == 0


In [ ]:
if not (REPO / ".git").is_dir():
    assert not REPO.exists(), "REPO tồn tại nhưng không phải Git repo."
    !git clone --no-checkout https://github.com/thulx18/PRDetect.git "{REPO}"
    assert get_ipython().user_ns.get("_exit_code", 1) == 0
    !git -C "{REPO}" checkout --detach {AUTHOR_COMMIT}
    assert get_ipython().user_ns.get("_exit_code", 1) == 0
else:
    # Không reset/checkout đè một repo đã dùng.
    import subprocess
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    assert actual == AUTHOR_COMMIT, f"Sai commit: {actual}; hãy dùng thư mục REPO mới."
for relative in ["gcn.py", "building_graph.py", "original_text/hc3_train.json",
                 "original_text/hc3_val.json", "original_text/gpt3.5_mixed_test_split.json"]:
    assert (REPO / relative).is_file(), relative
print("Author commit:", AUTHOR_COMMIT)


In [ ]:
%%writefile /kaggle/working/author_prdetect_gcn_exact/prepare_author_run.py
"""Prepare pinned author scripts, without rewriting the original training loop."""
import argparse
import difflib
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
from pathlib import Path

import spacy
import torch
from huggingface_hub import snapshot_download


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def replace_once(source, old, new):
    if source.count(old) != 1:
        raise RuntimeError("Pinned author source changed: " + repr(old))
    return source.replace(old, new, 1)


parser = argparse.ArgumentParser()
parser.add_argument("--repo", type=Path, required=True)
parser.add_argument("--run", type=Path, required=True)
parser.add_argument("--graph", type=Path, required=True)
parser.add_argument("--seed", type=int, default=2024)
args = parser.parse_args()
repo, run, graph = args.repo.resolve(), args.run.resolve(), args.graph.resolve()
commit = "83ebc6e2cd543477f83e405d702ac837e664fc9e"
revision = "e2da8e2f811d1448a5b465c236feacd80ffbac7b"
assert subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip() == commit
assert not subprocess.check_output(["git", "-C", str(repo), "diff", "--name-only"], text=True).strip()
assert platform.python_version_tuple()[:2] == ("3", "8")
assert torch.__version__ == "1.11.0+cu113"
expected = {
    "torch-geometric": "2.5.2", "torch-scatter": "2.0.9", "torch-sparse": "0.6.13",
    "transformers": "4.40.0", "huggingface-hub": "0.20.1", "spacy": "3.7.4",
    "en-core-web-sm": "3.7.1", "numpy": "1.22.4", "scipy": "1.10.1",
    "scikit-learn": "1.3.1", "tensorboard": "2.9.1",
    "protobuf": "3.19.6",
}
for package, version in expected.items():
    assert importlib.metadata.version(package) == version, (package, importlib.metadata.version(package))
assert torch.cuda.is_available(), "Enable a Kaggle T4/P100 GPU."
assert torch.cuda.get_device_capability()[0] < 9, "Use T4/P100; the author CUDA wheels predate this GPU."
run.mkdir(parents=True, exist_ok=True)
(run / "model").mkdir(exist_ok=True)
graph.mkdir(parents=True, exist_ok=True)
checkpoint = run / "model" / ("hc3_gcn_model_%s.pth" % args.seed)
assert not checkpoint.exists(), "Checkpoint exists. Choose a new RUN directory; do not overwrite it."

# The original builder loads a local RoBERTa, with no subword alignment replacement.
snapshot_download(
    "FacebookAI/roberta-base", revision=revision,
    local_dir=str(repo / "roberta-base"), local_dir_use_symlinks=False,
    allow_patterns=["config.json", "pytorch_model.bin", "vocab.json", "merges.txt", "tokenizer_config.json"],
)
source_train = (repo / "gcn.py").read_text(encoding="utf-8")
source_build = (repo / "building_graph.py").read_text(encoding="utf-8")
train = source_train.replace("/root/autodl-tmp/graph_data", graph.as_posix())
train = replace_once(train, "seed = 2024", "seed = %s" % args.seed)
train = replace_once(train, 'test_file = "hc3_test"', 'test_file = "gpt3.5_mixed_test_split"')
assert "optimizer = optim.Adam(gcnmodel.parameters(), lr=0.0001)" in train
assert "for i in tqdm(range(train_len)" in train
assert "epochs = 40" in train
assert "tag = 3" in train  # Preserve the author's ineffective early-stopping code.
build = source_build.replace("/root/autodl-tmp/graph_data", graph.as_posix())
build = replace_once(build, "import spacy\n", "import time\nimport spacy\n")
build = replace_once(build, '"hc3_test"', '"gpt3.5_mixed_test_split"')
# GPU encoder only; keep tokenization, graph edges and native GCN training unchanged.
build = replace_once(build, "vocab_size = len(tokenizer)",
                     'vocab_size = len(tokenizer)\n'
                     'assert torch.cuda.is_available(), "Enable a Kaggle GPU for graph building"\n'
                     'device = torch.device("cuda")\n'
                     'model = model.to(device)\nmodel.eval()\n'
                     'print("Graph encoder:", device, torch.cuda.get_device_name())')
# Never accumulate all document embeddings on the GPU or save CUDA tensors in pickle.
build = replace_once(build, 'chunk_outputs.append(token_embeddings)',
                     'chunk_outputs.append(token_embeddings.detach().cpu())')
# Exports run AFTER all native training/testing, and cannot affect the training RNG.
train_export = r'''

# Reporting-only addition after the unmodified native training and evaluation.
import hashlib as _hashlib
import json as _json
from pathlib import Path as _Path

_report_checkpoint = _Path("./model/%s_gcn_model_%s.pth" % (dataset_name, seed))
_truth = hc3_test["y"].cpu().numpy().astype(int).reshape(-1)
_pred = np.asarray(y_pred, dtype=int)
_matrix = [[int(((_truth == i) & (_pred == j)).sum()) for j in (0, 1)] for i in (0, 1)]
_history = [
    dict(epoch=i + 1, train_loss=float(train_loss[i]), train_accuracy=float(train_acc[i]),
         val_loss=float(val_loss[i]), val_accuracy=float(val_acc[i]))
    for i in range(len(val_acc))
]
_best = max(item["val_accuracy"] for item in _history)
_best_epoch = max(item["epoch"] for item in _history if item["val_accuracy"] == _best)
_report = dict(
    model="author_GCN2_native_training", seed=seed, train_batch_size=1, shuffle=False,
    optimizer="Adam", learning_rate=0.0001, weight_decay=0.0, dropout=0.5,
    epochs_requested=40, epochs_completed=len(_history), best_epoch=_best_epoch,
    best_validation_accuracy=_best, selection="validation accuracy; latest tied epoch",
    train_split="hc3_train", validation_split="hc3_val", test_split=test_file,
    samples=int(len(_truth)), threshold=0.5, loss=float(test_loss),
    accuracy=float(test_acc), auc=float(auc), f1_human=float(test_f1),
    recall_ai=_matrix[0][0] / sum(_matrix[0]),
    recall_human=_matrix[1][1] / sum(_matrix[1]),
    confusion_matrix_rows_true_columns_pred=_matrix, label_convention={"0": "ai", "1": "human"},
    y_true=_truth.tolist(), p_human=[float(value) for value in test_pres],
    checkpoint=str(_report_checkpoint.resolve()),
    checkpoint_sha256=_hashlib.sha256(_report_checkpoint.read_bytes()).hexdigest(),
)
assert len(_history) == 40
_Path("history.json").write_text(_json.dumps(_history, indent=2), encoding="utf-8")
_Path("gpt35_0pct_report.json").write_text(_json.dumps(_report, indent=2), encoding="utf-8")
print("\nAuthor-code reproduction finished:", {k: _report[k] for k in
      ("seed", "accuracy", "auc", "f1_human", "best_epoch")})
'''
build_export = r'''

# Integrity-only addition after the original graph builder.
import hashlib as _hashlib
import json as _json
from pathlib import Path as _Path

_manifest = {}
for _split, _expected_count in (("hc3_train", 8000), ("hc3_val", 1000),
                                ("gpt3.5_mixed_test_split", 1000)):
    _path = _Path(GRAPH_PATH_PLACEHOLDER) / (_split + ".pkl")
    with _path.open("rb") as _stream:
        _data = pickle.load(_stream)
    assert len(_data["y"]) == len(_data["all_token_embeddings"]) == len(_data["all_edge_index"]) == _expected_count
    for _x, _edge in zip(_data["all_token_embeddings"], _data["all_edge_index"]):
        assert _x.ndim == 2 and _x.shape[1] == 768 and _x.device.type == "cpu"
        assert _edge.shape[0] == 2 and int(_edge.min()) >= 0 and int(_edge.max()) < len(_x)
    _labels = _data["y"].tolist()
    assert set(_labels) == {0.0, 1.0}
    _node_count = sum(len(_x) for _x in _data["all_token_embeddings"])
    del _data, _x, _edge
    _digest = _hashlib.sha256()
    with _path.open("rb") as _stream:
        for _chunk in iter(lambda: _stream.read(1024 * 1024), b""):
            _digest.update(_chunk)
    _manifest[_split] = dict(samples=_expected_count, nodes=_node_count,
                            ai=_labels.count(0.0), human=_labels.count(1.0),
                            bytes=_path.stat().st_size, sha256=_digest.hexdigest())
_Path(MANIFEST_PATH_PLACEHOLDER).write_text(_json.dumps(_manifest, indent=2), encoding="utf-8")
print("Graph integrity OK:", _manifest)
'''
build_export = build_export.replace("GRAPH_PATH_PLACEHOLDER", repr(graph.as_posix()))
build_export = build_export.replace("MANIFEST_PATH_PLACEHOLDER", repr((run / "graph_manifest.json").as_posix()))
compile(train + train_export, "author_gcn_portable.py", "exec")
compile(build + build_export, "author_building_graph_portable.py", "exec")
(run / "author_gcn_portable.py").write_text(train + train_export, encoding="utf-8")
(run / "author_building_graph_portable.py").write_text(build + build_export, encoding="utf-8")
(run / "author_gcn_original.py").write_text(source_train, encoding="utf-8")
(run / "author_building_graph_original.py").write_text(source_build, encoding="utf-8")
diff = "".join(difflib.unified_diff(source_train.splitlines(True), (train + train_export).splitlines(True),
                                    fromfile="author/gcn.py", tofile="portable/gcn.py"))
diff += "".join(difflib.unified_diff(source_build.splitlines(True), (build + build_export).splitlines(True),
                                     fromfile="author/building_graph.py", tofile="portable/building_graph.py"))
(run / "compatibility_changes.diff").write_text(diff, encoding="utf-8")
native_files = ["gcn.py", "building_graph.py", "model/GCN2.py", "requirements.txt",
                "original_text/hc3_train.json", "original_text/hc3_val.json",
                "original_text/gpt3.5_mixed_test_split.json"]
weights_files = ["config.json", "vocab.json", "merges.txt", "pytorch_model.bin"]
manifest = dict(
    repository="https://github.com/thulx18/PRDetect", author_commit=commit,
    source_sha256={name: sha256(repo / name) for name in native_files},
    roberta_repository="FacebookAI/roberta-base", roberta_revision=revision,
    roberta_sha256={name: sha256(repo / "roberta-base" / name) for name in weights_files},
    seed=args.seed, epochs=40, train_batch_size=1, shuffle=False, learning_rate=1e-4,
    weight_decay=0.0, encoder_device="cuda", encoder_mode="eval",
    encoder_precision="float32", graph_storage_device="cpu", spacy_device="cpu",
    python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(), platform=platform.platform(),
    packages={name: importlib.metadata.version(name) for name in expected},
    spacy_model=spacy.load("en_core_web_sm").meta["version"],
    adaptations=["configurable graph paths", "add missing builder import time", "RoBERTa CUDA float32 inference; chunk outputs stored on CPU",
                 "test GPT-3.5 clean rather than HC3", "reporting-only exports after native computation"],
    limitation="Author graph caches, exact spaCy model release and encoder checkpoint hashes were not published; bitwise/paper-score equality is not guaranteed.",
)
(run / "run_config.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(run / "packages.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8",
)
print("Pinned author scripts ready:", run)
print("Train: native GCN2 / batch 1 / fixed order / Adam 1e-4 / weight_decay 0 / 40 epochs")


In [ ]:
!{PYTHON} "{PREPARE}" --repo "{REPO}" --run "{RUN}" --graph "{GRAPH}" --seed {SEED}
assert get_ipython().user_ns.get("_exit_code", 1) == 0, "Chưa đạt điều kiện môi trường/source tác giả."
config = json.loads((RUN / "run_config.json").read_text(encoding="utf-8"))
print({key: config[key] for key in ["author_commit", "seed", "train_batch_size", "shuffle",
                                    "weight_decay", "epochs", "encoder_device", "gpu"]})


In [ ]:
# Chính builder tác giả: 512 token spaCy/chunk, convert_tokens_to_ids.
expected_graphs = {"hc3_train.pkl", "hc3_val.pkl", "gpt3.5_mixed_test_split.pkl"}
assert len({path.name for path in GRAPH.glob("*.pkl")} | expected_graphs) <= 4
%cd {REPO}
!{PYTHON} "{BUILD_SCRIPT}"
assert get_ipython().user_ns.get("_exit_code", 1) == 0, "Build graph lỗi: không tiếp tục train."
graph_manifest = json.loads((RUN / "graph_manifest.json").read_text(encoding="utf-8"))
print({split: {"samples": item["samples"], "GiB": round(item["bytes"] / 2**30, 2)}
       for split, item in graph_manifest.items()})


In [ ]:
# Không dùng gcn.py nhóm, không nạp checkpoint Kaggle Input.
# Script native đặt torch seed đúng vị trí ngay trước khởi tạo GCN.
assert not (RUN / "model" / f"hc3_gcn_model_{SEED}.pth").exists(), "Không ghi đè checkpoint cũ."
assert (RUN / "graph_manifest.json").is_file()
%cd {RUN}
!{PYTHON} "{TRAIN_SCRIPT}"
assert get_ipython().user_ns.get("_exit_code", 1) == 0, "Native author training/test failed."
assert (RUN / "gpt35_0pct_report.json").is_file()


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

report = json.loads((RUN / "gpt35_0pct_report.json").read_text(encoding="utf-8"))
for key in ["seed", "samples", "train_batch_size", "epochs_completed", "best_epoch",
            "best_validation_accuracy", "accuracy", "auc", "f1_human", "recall_ai", "recall_human"]:
    print(f"{key}: {report[key]}")

ARCHIVE = WORK / f"author_gcn_hc3_to_gpt35_0pct_seed{SEED}.zip"
# Không zip graph/venv/RoBERTa.
included = ["run_config.json", "graph_manifest.json", "packages.txt", "history.json",
            "gpt35_0pct_report.json", "test_result.txt", "compatibility_changes.diff",
            "author_gcn_original.py", "author_gcn_portable.py",
            "author_building_graph_original.py", "author_building_graph_portable.py",
            f"model/hc3_gcn_model_{SEED}.pth"]
with ZipFile(ARCHIVE, "w", ZIP_DEFLATED) as archive:
    for relative in included:
        path = RUN / relative
        assert path.is_file(), relative
        archive.write(path, relative)
print("ZIP:", ARCHIVE)
print("Download this ZIP from the Kaggle Output panel.")
